# Music Annotation Project Notebook 91
## A5-v1 Freeze and Operational Bundle

**Runtime: GPU.** Select a T4 GPU and run the notebook in order.

This notebook converts the selected post-development A5 policy into one immutable candidate bundle. It does not inspect or score an untouched independent cohort.

**Frozen policy:** `A5-v1-print-p0980-margin0960-run2`

- final model: unchanged V8.2 compact 33x33 CNN refitted once on all permitted development samples;
- calibration: isotonic mapping fitted from saved source-group OOF raw scores and development truth;
- PRINT: calibrated probability >= 0.980 and binary margin >= 0.960;
- ANNOTATION: probability of annotation >= 0.995;
- minimum connected PRINT run: 2 pixels;
- assembly: start from A1 and copy exact observed bytes only at final PRINT coordinates;
- all other coordinates remain exact A1;
- clean truth is prohibited from the frozen runtime bundle.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow torch torchvision scikit-learn joblib


## 1. Configuration and change controls

Run twice unchanged with `CREATE_FREEZE=False`. Only after the same candidate lock appears twice should `CREATE_FREEZE=True` be set.


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, random, gc, sys, platform, shutil, time, csv
import cv2, torch, joblib
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.isotonic import IsotonicRegression

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_ROOT=ROOT/'observed_print_classifier_v8_2'
FREEZE_ROOT=ROOT/'a5_v1_freeze'
SOURCE_V82_RUN_OVERRIDE=None
CREATE_FREEZE=False
EXPECTED_CANDIDATE_LOCK_ID=None

POLICY_ID='A5-v1-print-p0980-margin0960-run2'
SEED=42; PATCH=33; HALF=16; BATCH=512; EPOCHS=12; LR=1e-3; WEIGHT_DECAY=1e-4
PRINT_THRESHOLD=0.980; PRINT_MARGIN=0.960; ANNOTATION_THRESHOLD=0.995; MIN_PRINT_RUN=2
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Notebook 91 requires a GPU runtime.')
torch.use_deterministic_algorithms(True); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def require(ok,msg):
    if not ok: raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def object_hash(value): return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def reset_seed():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
def complete(p):
    req=['experiment_identity.json','inputs/channel_schema.json','tables/stable_sample_manifest.csv','tables/stable_raw_oof.npy','tables/all_eligible_oof_predictions.parquet','audit/model_inventory.csv','audit/calibrator_inventory.csv']
    return p.is_dir() and all((p/x).exists() for x in req)
if SOURCE_V82_RUN_OVERRIDE:
    SOURCE=Path(SOURCE_V82_RUN_OVERRIDE)
else:
    hits=[p for p in SOURCE_ROOT.iterdir() if complete(p)] if SOURCE_ROOT.exists() else []
    require(hits,'No complete V8.2 source run found.')
    SOURCE=max(hits,key=lambda p:p.stat().st_mtime)
source_identity=json.loads((SOURCE/'experiment_identity.json').read_text())
require(source_identity.get('version')=='8.2-development-whole-page','Unexpected source version.')
require(source_identity.get('independent_data_used') is False,'Source must be development-only.')
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_a5_v1_freeze_candidate'
WORK=FREEZE_ROOT/'candidates'/RUN_ID
ARTEFACTS=WORK/'artefacts'; REPLAY=WORK/'replay'; AUDIT=WORK/'audit'; INPUTS=WORK/'inputs'; UPLOAD=WORK/'upload'
for p in [ARTEFACTS,REPLAY,AUDIT,INPUTS,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Source:',SOURCE); print('Candidate:',WORK); print('GPU:',torch.cuda.get_device_name(0))


## 2. Resolve the exact V8.2 development evidence

The stable sample roster supplies the permitted final-training rows. Saved source-group OOF raw scores supply calibration evidence. The source development truth is permitted here but is not copied into the operational runtime package.


In [ ]:
stable=pd.read_csv(SOURCE/'tables'/'stable_sample_manifest.csv')
full=pd.read_parquet(SOURCE/'tables'/'all_eligible_oof_predictions.parquet')
raw_oof=np.load(SOURCE/'tables'/'stable_raw_oof.npy')
required={'page','source_group','y','x','truth'}
require(required.issubset(stable.columns),f'Stable roster missing {sorted(required-set(stable.columns))}')
require(len(raw_oof)==len(stable),'Stable raw OOF length mismatch.')
require(np.isfinite(raw_oof).all(),'Stable OOF raw scores are non-finite.')
require(stable.truth.isin([0,1]).all(),'Final training roster must contain only ANNOTATION=0 and PRINT=1.')
require(not stable.duplicated(['page','y','x']).any(),'Duplicate stable centres.')
channel_schema=json.loads((SOURCE/'inputs'/'channel_schema.json').read_text())
require(channel_schema.get('patch')==PATCH,'Patch mismatch.')
require(channel_schema.get('channels')==CHANNELS,'Channel order mismatch.')
stable.to_csv(INPUTS/'training_roster.csv',index=False)
pd.DataFrame({'sample_id':np.arange(len(stable)),'page':stable.page,'source_group':stable.source_group,'y':stable.y,'x':stable.x,'truth':stable.truth,'raw_oof':raw_oof}).to_csv(INPUTS/'calibration_roster.csv',index=False)
shutil.copy2(SOURCE/'inputs'/'channel_schema.json',ARTEFACTS/'channel_schema.json')
print('Training rows:',len(stable),'pages:',stable.page.nunique(),'groups:',stable.source_group.nunique())


## 3. Rebuild observable tensors from the exact V8.2 reference

Clean pages are loaded only to reproduce the selected development roster and are not included in the frozen runtime package.


In [ ]:
v75=Path(source_identity['v7_5_reference'])
status=pd.read_csv(v75/'tables'/'cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
observed_map=image_map(ROOT/'aligned')
pages=sorted(stable.page.astype(str).unique())
required_arrays=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
runtime={}; padded={}; page_audit=[]
def background_surface(obs,mask,sigma=45.0):
    visible=(~mask)&(obs>=200); num=cv2.GaussianBlur(obs.astype(np.float32)*visible,(0,0),sigma); den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma); fallback=cv2.GaussianBlur(obs,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
for page in pages:
    require(page in cache and page in observed_map,f'{page}: missing V7.5 cache or observed page.')
    with np.load(cache[page]) as z:
        miss=[k for k in required_arrays if k not in z.files]; require(not miss,f'{page}: cache missing {miss}')
        a={k:z[k].copy() for k in required_arrays}
    obs=cv2.imread(str(observed_map[page]),0); require(obs is not None,f'{page}: observed unreadable.')
    require(all(v.shape==obs.shape for v in a.values()),f'{page}: array shape mismatch.')
    mask=a['mask']>0; paper=background_surface(obs,mask); norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
    runtime[page]={'obs':obs,'norm':norm,'mask':mask,'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),'edge':cv2.Canny(obs,70,170)>0,'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'ambiguous':a['g80_ambiguous']>0,'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0,'a1':a['geometry_t80']}
    padded[page]={}
    for k in CHANNELS:
        v=runtime[page][k].astype(np.float32)
        if k in ['obs','norm']: v/=255.0
        padded[page][k]=np.pad(v,HALF,mode='reflect')
    page_audit.append({'page':page,'observed_sha256':sha256_file(observed_map[page]),'cache_sha256':sha256_file(cache[page]),'shape':str(obs.shape),'a1_array_sha256':array_hash(a['geometry_t80'])})
pd.DataFrame(page_audit).to_csv(AUDIT/'page_input_audit.csv',index=False)
page_list=pages; page_index={p:i for i,p in enumerate(page_list)}
pids=stable.page.astype(str).map(page_index).to_numpy(np.int16); ys=stable.y.to_numpy(np.int32); xs=stable.x.to_numpy(np.int32); truth=stable.truth.to_numpy(np.int8)
def get_patch(pid,y,x):
    page=page_list[int(pid)]; return np.stack([padded[page][k][y:y+PATCH,x:x+PATCH] for k in CHANNELS]).astype(np.float32)
print('Observable tensors ready.')


## 4. Final all-development model fit


In [ ]:
class PatchDataset(Dataset):
    def __len__(self): return len(truth)
    def __getitem__(self,i): return torch.from_numpy(get_patch(pids[i],ys[i],xs[i])),torch.tensor(int(truth[i]),dtype=torch.long)
class TopologyCNN(nn.Module):
    def __init__(self,nc=len(CHANNELS)):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,2)
    def forward(self,x): return self.head(self.net(x).flatten(1))
def worker_init(w): random.seed(SEED+w); np.random.seed(SEED+w)
g=torch.Generator(); g.manual_seed(SEED)
loader=DataLoader(PatchDataset(),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True,generator=g,worker_init_fn=worker_init,persistent_workers=False)
reset_seed(); model=TopologyCNN().to(DEVICE); counts=np.bincount(truth,minlength=2); weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE); loss_fn=nn.CrossEntropyLoss(weight=weights); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
history=[]
for epoch in range(EPOCHS):
    model.train(); total=0.0
    for xb,yb in loader:
        xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); loss=loss_fn(model(xb),yb); loss.backward(); opt.step(); total+=float(loss.detach().cpu())*len(yb)
    mean=total/len(stable); history.append({'epoch':epoch+1,'mean_loss':mean}); print(f'epoch {epoch+1:02d}/{EPOCHS}: {mean:.6f}')
model_path=ARTEFACTS/'model.pt'; torch.save(model.state_dict(),model_path)
model_config={'class':'TopologyCNN','channels':CHANNELS,'patch':PATCH,'seed':SEED,'epochs':EPOCHS,'batch':BATCH,'lr':LR,'weight_decay':WEIGHT_DECAY,'class_weight_rule':'n_total/n_class','training_rows':len(stable),'training_roster_sha256':sha256_file(INPUTS/'training_roster.csv')}
atomic_json(ARTEFACTS/'model_config.json',model_config); pd.DataFrame(history).to_csv(AUDIT/'training_history.csv',index=False)
print('Model SHA-256:',sha256_file(model_path))


## 5. Final calibrator and frozen policy

The calibrator is fitted only from saved source-group OOF raw scores and development labels. No full-fit training prediction is used to fit calibration.


In [ ]:
calibrator=IsotonicRegression(out_of_bounds='clip').fit(raw_oof,truth)
cal_path=ARTEFACTS/'isotonic_calibrator.joblib'; joblib.dump(calibrator,cal_path)
cal_semantic={'x_thresholds':calibrator.X_thresholds_.tolist(),'y_thresholds':calibrator.y_thresholds_.tolist(),'method':'IsotonicRegression','out_of_bounds':'clip','fit_source':'V8.2 source-group OOF stable raw predictions','rows':len(stable),'calibration_roster_sha256':sha256_file(INPUTS/'calibration_roster.csv')}
atomic_json(ARTEFACTS/'calibrator.json',cal_semantic)
policy={'policy_id':POLICY_ID,'status':'FROZEN_CANDIDATE_FOR_INDEPENDENT_TEST','print_threshold':PRINT_THRESHOLD,'print_margin':PRINT_MARGIN,'effective_print_threshold':max(PRINT_THRESHOLD,(1+PRINT_MARGIN)/2),'annotation_threshold':ANNOTATION_THRESHOLD,'minimum_print_run':MIN_PRINT_RUN,'connectivity':8,'truth_codes':{'0':'ANNOTATION','1':'PRINT','2':'UNCERTAIN'},'print_action':'COPY_EXACT_OBSERVED_BYTE','annotation_action':'KEEP_A1_BYTE','uncertain_action':'KEEP_A1_BYTE','fallback':'A1','outside_authoritative_mask':'IMMUTABLE_OBSERVED','runtime_truth_permitted':False,'clean_image_permitted_at_runtime':False,'raw_page_coordinates_as_model_inputs':False,'development_policy_selected_post_outcome':True,'independent_validation_completed':False,'production_authorised':False}
atomic_json(ARTEFACTS/'policy.json',policy)
print('Calibrator SHA-256:',sha256_file(cal_path)); print(json.dumps(policy,indent=2))


## 6. Write the truth-free inference runtime


In [ ]:
runtime_source=r"""import json, hashlib
from pathlib import Path
import cv2, joblib, numpy as np, torch
from torch import nn

class TopologyCNN(nn.Module):
    def __init__(self,nc=11):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,2)
    def forward(self,x): return self.head(self.net(x).flatten(1))

def load_bundle(bundle_dir,device=None):
    bundle=Path(bundle_dir); policy=json.loads((bundle/'policy.json').read_text()); schema=json.loads((bundle/'channel_schema.json').read_text()); config=json.loads((bundle/'model_config.json').read_text()); device=device or ('cuda' if torch.cuda.is_available() else 'cpu'); model=TopologyCNN(len(schema['channels'])).to(device); model.load_state_dict(torch.load(bundle/'model.pt',map_location=device)); model.eval(); calibrator=joblib.load(bundle/'isotonic_calibrator.joblib'); return model,calibrator,policy,schema,device

def make_padded(channels,schema):
    half=int(schema['patch'])//2; out={}
    for name in schema['channels']:
        if name not in channels: raise RuntimeError(f'Missing channel: {name}')
        a=np.asarray(channels[name]);
        if a.ndim!=2: raise RuntimeError(f'{name}: expected 2D array')
        v=a.astype(np.float32)
        if name in ('obs','norm'): v/=255.0
        if not np.isfinite(v).all(): raise RuntimeError(f'{name}: non-finite values')
        out[name]=np.pad(v,half,mode='reflect')
    shapes={np.asarray(channels[n]).shape for n in schema['channels']}
    if len(shapes)!=1: raise RuntimeError(f'Channel shape mismatch: {shapes}')
    return out

def score_coordinates(model,calibrator,padded,schema,ys,xs,device,batch=512):
    ys=np.asarray(ys,dtype=np.int32); xs=np.asarray(xs,dtype=np.int32); patch=int(schema['patch']); probs=[]
    with torch.no_grad():
        for start in range(0,len(ys),batch):
            stop=min(len(ys),start+batch); tensors=[]
            for y,x in zip(ys[start:stop],xs[start:stop]): tensors.append(np.stack([padded[n][y:y+patch,x:x+patch] for n in schema['channels']]))
            xb=torch.from_numpy(np.asarray(tensors,dtype=np.float32)).to(device); raw=torch.softmax(model(xb),1)[:,1].cpu().numpy(); probs.extend(calibrator.predict(raw))
    return np.asarray(probs,dtype=np.float64)

def decisions(prob,policy,shape,ys,xs):
    p=np.asarray(prob,dtype=np.float64); q=1-p; out=np.full(len(p),'UNCERTAIN',object); out[q>=policy['annotation_threshold']]='ANNOTATION'; out[(p>=policy['print_threshold'])&((p-q)>=policy['print_margin'])]='PRINT'
    provisional=out.copy(); pm=np.zeros(shape,np.uint8); idx=np.where(provisional=='PRINT')[0]; pm[np.asarray(ys)[idx],np.asarray(xs)[idx]]=1; n,lab,stats,_=cv2.connectedComponentsWithStats(pm,8)
    for cid in range(1,n):
        if stats[cid,cv2.CC_STAT_AREA] < int(policy['minimum_print_run']):
            yy,xx=np.where(lab==cid); keys=set(zip(yy.tolist(),xx.tolist())); drop=[i for i in idx if (int(ys[i]),int(xs[i])) in keys]; out[drop]='UNCERTAIN'
    return provisional,out

def assemble(observed,a1,authoritative_mask,ys,xs,final_decisions):
    observed=np.asarray(observed,dtype=np.uint8); a1=np.asarray(a1,dtype=np.uint8); mask=np.asarray(authoritative_mask)>0
    if not (observed.shape==a1.shape==mask.shape): raise RuntimeError('Observed/A1/mask shape mismatch')
    out=a1.copy(); idx=np.where(np.asarray(final_decisions)=='PRINT')[0]; py=np.asarray(ys)[idx]; px=np.asarray(xs)[idx]; out[py,px]=observed[py,px]
    print_mask=np.zeros(mask.shape,bool); print_mask[py,px]=True
    if np.any(out[~mask]!=observed[~mask]): raise RuntimeError('Outside-mask drift')
    if np.any(out[~print_mask]!=a1[~print_mask]): raise RuntimeError('Non-PRINT A1 regression')
    if np.any((out!=a1)&(out!=observed)): raise RuntimeError('Changed byte not equal observed')
    return out,print_mask
"""
(ARTEFACTS/'inference_runtime.py').write_text(runtime_source)
print('Runtime written:',ARTEFACTS/'inference_runtime.py')


## 7. Deterministic replay pack

Replay uses observable arrays only. Expected outputs are created once in this candidate run and then recomputed from reloaded artefacts.


In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('a5_runtime',ARTEFACTS/'inference_runtime.py'); rt=importlib.util.module_from_spec(spec); spec.loader.exec_module(rt)
# Select up to three pages spanning source groups and with sufficient eligible rows.
replay_pages=[]
for group,g in stable.groupby('source_group',sort=True):
    for page in sorted(g.page.astype(str).unique()):
        if len(full[full.page.astype(str)==page])>=20: replay_pages.append(page); break
    if len(replay_pages)>=3: break
require(replay_pages,'No suitable replay pages.')
model_loaded,cal_loaded,policy_loaded,schema_loaded,replay_device=rt.load_bundle(ARTEFACTS,DEVICE)
replay_records=[]
for page in replay_pages:
    rows=full[full.page.astype(str)==page].sort_values(['y','x']).head(5000)
    y=rows.y.to_numpy(np.int32); x=rows.x.to_numpy(np.int32); z=runtime[page]
    channels={k:z[k] for k in CHANNELS}; padded_page=rt.make_padded(channels,schema_loaded); p=rt.score_coordinates(model_loaded,cal_loaded,padded_page,schema_loaded,y,x,replay_device,BATCH); pre,final=rt.decisions(p,policy_loaded,z['obs'].shape,y,x); out,pm=rt.assemble(z['obs'],z['a1'],z['mask'],y,x,final)
    page_dir=REPLAY/page; page_dir.mkdir(parents=True,exist_ok=False)
    np.savez_compressed(page_dir/'observable_inputs.npz',**{k:channels[k] for k in CHANNELS},a1=z['a1'],y=y,x=x)
    np.save(page_dir/'expected_probability.npy',p); np.save(page_dir/'expected_provisional.npy',pre); np.save(page_dir/'expected_final.npy',final); cv2.imwrite(str(page_dir/'expected_a5.png'),out)
    replay_records.append({'page':page,'rows':len(rows),'probability_sha256':sha256_file(page_dir/'expected_probability.npy'),'final_sha256':sha256_file(page_dir/'expected_final.npy'),'a5_sha256':sha256_file(page_dir/'expected_a5.png')})
# Exact reload replay.
replay_checks=[]
model2,cal2,policy2,schema2,device2=rt.load_bundle(ARTEFACTS,DEVICE)
for rec in replay_records:
    page=rec['page']; d=REPLAY/page
    with np.load(d/'observable_inputs.npz') as q: data={k:q[k] for k in q.files}
    y=data.pop('y'); x=data.pop('x'); a1=data.pop('a1'); padded_page=rt.make_padded(data,schema2); p=rt.score_coordinates(model2,cal2,padded_page,schema2,y,x,device2,BATCH); pre,final=rt.decisions(p,policy2,a1.shape,y,x); out,pm=rt.assemble(data['obs'],a1,data['mask'],y,x,final)
    expected_p=np.load(d/'expected_probability.npy'); expected_final=np.load(d/'expected_final.npy',allow_pickle=True); expected_out=cv2.imread(str(d/'expected_a5.png'),0)
    check={'page':page,'probability_exact':bool(np.array_equal(p,expected_p)),'decision_exact':bool(np.array_equal(final,expected_final)),'page_exact':bool(np.array_equal(out,expected_out))}; replay_checks.append(check)
replay_passed=all(all(v for k,v in r.items() if k!='page') for r in replay_checks)
atomic_json(REPLAY/'replay_test.json',{'policy_id':POLICY_ID,'passed':replay_passed,'pages':replay_pages,'records':replay_records,'checks':replay_checks})
require(replay_passed,'Replay failed. Candidate cannot be frozen.')
print(json.dumps(replay_checks,indent=2))


## 8. Candidate lock and two-run freeze gate


In [ ]:
environment={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'joblib':joblib.__version__ if hasattr(joblib,'__version__') else None,'platform':platform.platform()}
atomic_json(ARTEFACTS/'software_environment.json',environment)
roles={'model.pt':'model','model_config.json':'model_config','isotonic_calibrator.joblib':'calibrator','calibrator.json':'calibrator_semantics','channel_schema.json':'channel_schema','policy.json':'policy','inference_runtime.py':'runtime','software_environment.json':'environment'}
manifest=[]
for p in sorted(ARTEFACTS.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':str(p.relative_to(ARTEFACTS)),'role':roles.get(p.name,'artefact'),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
for p in sorted(REPLAY.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':'replay/'+str(p.relative_to(REPLAY)),'role':'replay','bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(AUDIT/'artifact_manifest.csv',index=False)
lock_payload={'policy_id':POLICY_ID,'source_v82_run':str(SOURCE),'source_identity_sha256':sha256_file(SOURCE/'experiment_identity.json'),'source_predictions_sha256':sha256_file(SOURCE/'tables'/'all_eligible_oof_predictions.parquet'),'training_roster_sha256':sha256_file(INPUTS/'training_roster.csv'),'calibration_roster_sha256':sha256_file(INPUTS/'calibration_roster.csv'),'artifact_manifest_sha256':sha256_file(AUDIT/'artifact_manifest.csv'),'replay_test_sha256':sha256_file(REPLAY/'replay_test.json'),'policy_sha256':sha256_file(ARTEFACTS/'policy.json'),'channel_schema_sha256':sha256_file(ARTEFACTS/'channel_schema.json'),'model_sha256':sha256_file(ARTEFACTS/'model.pt'),'calibrator_sha256':sha256_file(ARTEFACTS/'isotonic_calibrator.joblib'),'runtime_sha256':sha256_file(ARTEFACTS/'inference_runtime.py'),'replay_passed':True}
CANDIDATE_LOCK_ID=object_hash(lock_payload)
FREEZE_ROOT.mkdir(parents=True,exist_ok=True); candidate_state=FREEZE_ROOT/'candidate_lock_state.json'
previous=json.loads(candidate_state.read_text()) if candidate_state.exists() else None
stable=bool(previous and previous.get('candidate_lock_id')==CANDIDATE_LOCK_ID and previous.get('payload')==lock_payload)
atomic_json(candidate_state,{'candidate_lock_id':CANDIDATE_LOCK_ID,'payload':lock_payload,'calculated_utc':datetime.now(timezone.utc).isoformat()})
atomic_json(WORK/'candidate_freeze_manifest.json',{'candidate_lock_id':CANDIDATE_LOCK_ID,'payload':lock_payload,'two_run_stable':stable,'create_freeze_requested':CREATE_FREEZE})
print('Candidate lock:',CANDIDATE_LOCK_ID); print('Two-run stable:',stable)
if previous is None: print('First candidate calculation. Run unchanged once more with CREATE_FREEZE=False.')
elif not stable: print('Candidate changed. Do not freeze; inspect source/environment determinism.')
else: print('Deterministic candidate check passed.')


## 9. Create immutable A5-v1 bundle

This cell refuses to freeze unless the candidate lock was reproduced unchanged and `EXPECTED_CANDIDATE_LOCK_ID` matches.


In [ ]:
if CREATE_FREEZE:
    require(stable,'Two consecutive unchanged candidate locks are required.')
    require(EXPECTED_CANDIDATE_LOCK_ID==CANDIDATE_LOCK_ID,'Register EXPECTED_CANDIDATE_LOCK_ID before freezing.')
    FINAL=FREEZE_ROOT/'frozen'/POLICY_ID/CANDIDATE_LOCK_ID
    require(not FINAL.exists(),f'Frozen bundle already exists: {FINAL}')
    FINAL.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(WORK,FINAL)
    final_manifest={'policy_id':POLICY_ID,'freeze_status':'FROZEN_CANDIDATE_FOR_INDEPENDENT_TEST','candidate_lock_id':CANDIDATE_LOCK_ID,'created_utc':datetime.now(timezone.utc).isoformat(),'payload':lock_payload,'independent_validation_completed':False,'production_authorised':False,'source_working_candidate':str(WORK)}
    atomic_json(FINAL/'freeze_manifest.json',final_manifest)
    print('FROZEN A5-v1 BUNDLE:',FINAL)
else:
    print('Freeze not created. Complete the two-run gate, register EXPECTED_CANDIDATE_LOCK_ID, then set CREATE_FREEZE=True.')


## 10. Final upload package

The review bundle contains policy, manifests, replay record, environment and compact audit records. The final trained checkpoint remains in the frozen bundle and is referenced by hash.


In [ ]:
readme="""A5-v1 FREEZE CANDIDATE REVIEW PACKAGE

Policy: A5-v1-print-p0980-margin0960-run2
Purpose: convert the selected post-development A5 policy into an immutable candidate bundle.
Status: candidate until the two-run lock is completed; frozen candidate only after CREATE_FREEZE succeeds.
Independent validation: not completed.
Production authorisation: no.
"""
(UPLOAD/'README.txt').write_text(readme)
curated=[ARTEFACTS/'policy.json',ARTEFACTS/'model_config.json',ARTEFACTS/'channel_schema.json',ARTEFACTS/'calibrator.json',ARTEFACTS/'software_environment.json',REPLAY/'replay_test.json',AUDIT/'artifact_manifest.csv',WORK/'candidate_freeze_manifest.json']
for src in curated:
    require(src.exists(),f'Missing upload artefact: {src}'); shutil.copy2(src,UPLOAD/src.name)
upload_rows=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file(): upload_rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(upload_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
shutil.make_archive(str(WORK/'notebook91_a5_v1_freeze_review_bundle'),'zip',UPLOAD)
print('Review bundle:',WORK/'notebook91_a5_v1_freeze_review_bundle.zip')


## Completion criteria

Notebook 91 is complete only when:

1. two consecutive unchanged candidate lock IDs have been obtained;
2. `CREATE_FREEZE=True` succeeds with the registered lock ID;
3. `freeze_manifest.json` exists in the immutable final directory;
4. replay probability, decisions and A5 page bytes all match exactly; and
5. the frozen bundle is used unchanged by the untouched independent-test notebook.

**Next question:** Does the untouched independent cohort sustain A5-v1's 0.980 safety/coverage balance without concentrated annotation-retention failures?
